In [145]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
from collections import Counter

In [146]:
train = pd.read_csv('data/train.csv',sep=";")

In [147]:
labels = train['label'].to_list()
texts = train['text'].to_list()

In [148]:
#Não posso passar texto pro LSTM - Vamos criar uma vetorização simples

def build_vocab(texts,max_vocab_size = 10000):
 
    word_counter = Counter()
    for text in texts:
        # Fazendo uma tokenização super simples para desenhar o framework final.
        tokens = text.lower().split()
        word_counter.update(tokens)

    most_relevant = word_counter.most_common(max_vocab_size)

    # Porque eu precisaria fazer padding e 0 vai ser usado.
    vocab = {word:idx+1 for idx, (word,freq) in enumerate(most_relevant)}

    return vocab    

In [149]:
vocab = build_vocab(texts)

In [150]:
# Tentando estimar o tamanho médio ou 95%

train['tokens_len'] = train['text'].apply(lambda x: len(x.lower().split()))
train['tokens_len'] 


0        212
1         59
2        354
3        428
4        314
        ... 
24348    240
24349    248
24350    302
24351    438
24352    317
Name: tokens_len, Length: 24353, dtype: int64

In [151]:
train['tokens_len'].describe(percentiles=[0.5,0.9,0.95])

count    24353.000000
mean       410.985300
std        345.329248
min          0.000000
50%        369.000000
90%        750.000000
95%        904.400000
max       7928.000000
Name: tokens_len, dtype: float64

In [ ]:
### Criar uma classe para preparar os dados!
### Recebo o texto e os label, retorno os tensores respectivos já normalizados no tamanho padrão!

class DataLSTM(Dataset):
    def __init__(self, texts, labels, vocab, norm_len=900):

        # texts: natural a sentença.
        # labels
        # norm_len = comprimento de tokens por sequencia menos eu vou aplicar padding, mais eu aplico truncate.

        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.norm_len = norm_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self,idx):

        text = self.texts[idx]
        label = self.labels[idx]
        
        # tratar o text gerando tokens
        tokens = text.lower().split()
        # De word -> idx, se eu nao sei o vocab eu ignoro
        tokens = [self.vocab.get(token,0) for token in tokens]

        tokens = tokens[:self.norm_len]
        # Se precisar aplicar o padding
        if len(tokens) < self.norm_len:
            tokens = tokens + [0]*(self.norm_len-len(tokens))
        
        return torch.tensor(tokens,dtype=torch.long), torch.tensor(label,dtype=torch.float)



In [153]:
# 1 - construimos nossa classe agora preparamos ela para treino!
train_dataset = DataLSTM(texts=texts, labels=labels, vocab=vocab) # ja defini que a dimen máx é 900

# 2 - Instanciando o Loader! Agora podemos ir pro modelo.
train_loader = DataLoader(
    dataset=train_dataset, 
    batch_size=64,      # Baixamos para testar
    shuffle=True,      # misturar entrada
)

In [154]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100,hidden_dim=128): 
        super().__init__() # vou pergar tudo do nn.Module.
        ## no init vamos criar os layers.

        self.embedding = nn.Embedding(vocab_size + 1, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim,1)

    def forward(self, x):
        #executar embedding
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        # Pegamos a saída do último estado oculto e jogamos no .fc
        return self.fc(hidden[-1]) # Saída do último estado oculto

In [156]:
# Configuração de dispositivo (usa GPU se disponível, senão CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Pipeline de Treino!

## Treino do modelo

model = LSTM(vocab_size=len(vocab.keys()))
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 5

for epoch in tqdm.tqdm(range(epochs)):

    model.train()
    total_loss = 0

    for batch_id,(tokens,labels) in enumerate(train_loader):
        
        #print(batch_id)
        #zerando os gradientes:
        optimizer.zero_grad()
        # Fazendo predict:
        outputs = model(tokens)
        
        # loss para novo predict 
        loss = criterion(outputs.squeeze(1), labels)

        # propagar a loss para atualizar o peso.
        loss.backward()

        # Ajusta os pesos
        optimizer.step()
        
        total_loss += loss.item()

    print(f'{epoch}:{total_loss}')

 20%|██        | 1/5 [00:08<00:35,  8.78s/it]

0:239.76513221859932


 40%|████      | 2/5 [00:17<00:25,  8.50s/it]

1:85.33477791585028


 60%|██████    | 3/5 [00:25<00:16,  8.42s/it]

2:39.99464623071253


 80%|████████  | 4/5 [00:33<00:08,  8.37s/it]

3:31.440436673816293


100%|██████████| 5/5 [00:42<00:00,  8.42s/it]

4:28.76617344841361


In [159]:
test = pd.read_csv('data/test.csv',sep=";")

In [165]:
# Predict and calculate metrics:

test_dataset = DataLSTM(texts = test['text'].to_list(), labels = test['label'].to_list(),vocab=vocab )
test_loader = DataLoader(
    dataset = test_dataset,
    batch_size=32)

In [168]:
from sklearn.metrics import f1_score


In [181]:
labels_list = []
pred_label = []

model.eval() # Modo avaliação

with torch.no_grad(): # Desliga o cálculo de gradientes (mais rápido e gasta menos memória)
    for tokens, labels in test_loader:
        tokens = tokens.to(device)

        outputs = model(tokens)
        # Transforma Logit em Probabilidade (0-1) e depois em Classe (0 ou 1)
        preds = (torch.sigmoid(outputs.squeeze(1)) > 0.5).int()

        pred_label.extend(preds.tolist())
        labels_list.extend(labels.tolist())


In [187]:
results = pd.DataFrame({'preds':pred_label,'true':labels_list})

In [188]:
f1_score(results['true'],results['preds'])

0.968895978124644